# Fine-tuning local con Unsloth: Qwen3.5-4B y Gemma 4 E2B

**Kai — La comunidad AI de LATAM** · [kaizen.la](https://kaizen.la)

Este notebook implementa fine-tuning LoRA de dos modelos sobre un **T4 de Google Colab gratuito** (15GB VRAM). El resultado es un archivo GGUF que podés correr offline en tu Mac M1, Ollama o Unsloth Studio.

---

## Requisitos
- Colab con GPU T4 (`Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`)
- Para Qwen3.5-4B: ~10GB VRAM (bf16 LoRA) → T4 tiene 15GB ✅
- Para Gemma 4 E2B: ~8-10GB VRAM (4-bit LoRA) → T4 ✅

---

## Elegí tu modelo

| Modelo | VRAM | Caso de uso |
|--------|------|-------------|
| **Qwen3.5-4B** | ~10GB (bf16) | Multilingüe, muy bueno en español |
| **Gemma 4 E2B** | ~8GB (4-bit) | Compacto, visión opcional, eficiente |

## 0. Verificar GPU disponible

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("⚠️  No hay GPU. Andá a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu}")
print(f"VRAM disponible: {vram_gb:.1f} GB")
print(f"CUDA version: {torch.version.cuda}")

## 1. Instalar Unsloth y dependencias

> ⚠️ `transformers >= 5.0` es obligatorio para Qwen3.5. Versiones anteriores fallan silenciosamente.

In [ ]:
%%capture
# Instalar Unsloth para Colab con CUDA 12.x
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.9.0" "peft>=0.11.0" "accelerate>=0.34.0"
!pip install "transformers>=5.0"  # Requerido para Qwen3.5
!pip install datasets

print("✅ Instalación completa")

## 2A. Cargar modelo: Qwen3.5-4B (recomendado para español)

**Nota importante**: Para Qwen3.5, usar `load_in_16bit=True`. El QLoRA (4-bit) **no está recomendado** por Unsloth para esta familia de modelos porque afecta la calidad del fine-tuning.

In [ ]:
# ─── ELEGÍ UN MODELO (descomentar uno) ────────────────────────────────────
MODELO = "qwen35"  # "qwen35" o "gemma4"
# ──────────────────────────────────────────────────────────────────────────

MAX_SEQ_LENGTH = 2048

if MODELO == "qwen35":
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3.5-4B-Instruct",
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = False,
        load_in_16bit = True,   # ← bf16, NO 4-bit para Qwen3.5
        full_finetuning = False,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",  # Reduce VRAM, extiende contexto
        random_state = 3407,
        max_seq_length = MAX_SEQ_LENGTH,
    )

    print("✅ Qwen3.5-4B-Instruct cargado con LoRA r=16")

elif MODELO == "gemma4":
    from unsloth import FastModel

    model, tokenizer = FastModel.from_pretrained(
        model_name = "unsloth/gemma-4-E2B-it",
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = True,    # ← 4-bit OK para Gemma 4
        full_finetuning = False,
    )

    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers = False,
        finetune_language_layers = True,
        finetune_attention_modules = True,
        finetune_mlp_modules = True,
        r = 8,
        lora_alpha = 8,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
    )

    print("✅ Gemma 4 E2B cargado con LoRA r=8")

## 3. Dataset

Tres opciones:

- **Opción A**: Dataset de demostración (en inglés, carga inmediata) — para probar que el pipeline funciona
- **Opción B**: Tu propio dataset JSONL desde Google Drive
- **Opción C**: Dataset en español desde HuggingFace

El formato esperado es **ShareGPT / chat** con `conversations` (lista de turnos user/assistant).

In [ ]:
from datasets import load_dataset

# ─── ELEGÍ UNA OPCIÓN ─────────────────────────────────────────────────────
OPCION_DATASET = "A"  # "A", "B" o "C"
# ──────────────────────────────────────────────────────────────────────────

if OPCION_DATASET == "A":
    # Demo: chip2 (inglés, instrucción-respuesta)
    url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
    dataset = load_dataset("json", data_files={"train": url}, split="train[:500]")
    # Este dataset tiene columnas 'text' con formato "Human: ... Assistant: ..."
    dataset_text_field = "text"
    print(f"✅ Dataset demo cargado: {len(dataset)} ejemplos")
    print(f"Ejemplo: {dataset[0]['text'][:200]}...")

elif OPCION_DATASET == "B":
    # Tu propio JSONL desde Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    RUTA_DRIVE = "/content/drive/MyDrive/mi-dataset.jsonl"  # ← cambiá esto
    dataset = load_dataset("json", data_files={"train": RUTA_DRIVE}, split="train")
    dataset_text_field = "text"  # ← cambiá según tu formato
    print(f"✅ Dataset propio cargado: {len(dataset)} ejemplos")

elif OPCION_DATASET == "C":
    # Dataset en español (instrucción-respuesta)
    dataset = load_dataset("bertin-project/alpaca-spanish", split="train[:1000]")
    # Convertir al formato texto plano
    def format_alpaca_es(example):
        return {"text": f"### Instrucción:\n{example['instruction']}\n\n### Respuesta:\n{example['output']}"}
    dataset = dataset.map(format_alpaca_es)
    dataset_text_field = "text"
    print(f"✅ Dataset en español cargado: {len(dataset)} ejemplos")

## 4. Entrenamiento

Parámetros conservadores para T4 de Colab gratuito. Si querés más épocas, reemplazá `max_steps` por `num_train_epochs`.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = dataset_text_field,
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,  # Effective batch = 4
        warmup_steps = 10,
        max_steps = 100,              # Para demo (~5 min en T4)
        # num_train_epochs = 1,       # ← Alternativa: épocas completas
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = f"outputs_{MODELO}",
        dataset_num_proc = 2,
        report_to = "none",           # Cambiar a "wandb" si usás W&B
    ),
)

print("Iniciando entrenamiento...")
print(f"Steps totales: {trainer.args.max_steps}")

trainer_stats = trainer.train()

print(f"\n✅ Entrenamiento completo")
print(f"Tiempo total: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Loss final: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Test rápido del modelo entrenado

In [ ]:
from unsloth.chat_templates import get_chat_template

if MODELO == "qwen35":
    FastLanguageModel.for_inference(model)  # Activar modo inferencia (2x más rápido)
    template = "qwen-2.5"
elif MODELO == "gemma4":
    FastModel.for_inference(model)
    template = "gemma-4"

tokenizer = get_chat_template(tokenizer, chat_template=template)

messages = [
    {"role": "user", "content": "Explicá en 3 pasos cómo funciona LoRA para fine-tuning de LLMs."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

respuesta = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=" * 60)
print(respuesta)
print("=" * 60)

## 6. Exportar a GGUF

El GGUF resultante lo podés:
- Bajar a tu Mac y cargarlo en **Ollama** o **Unsloth Studio**
- Subir a **HuggingFace Hub** y abrirlo desde Studio directamente

> **Formatos disponibles**: `q4_k_m` (recomendado, balance calidad/tamaño), `q8_0` (mayor calidad), `f16` (sin cuantización, más pesado)

In [ ]:
import os

OUTPUT_DIR = f"kai-{MODELO}-finetuned"
QUANTIZATION = "q4_k_m"  # ← cambiá si querés q8_0 o f16

print(f"Exportando a GGUF ({QUANTIZATION})...")
model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method=QUANTIZATION)

# Listar archivos generados
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024**2
    print(f"  {f}  ({size_mb:.0f} MB)")

print(f"\n✅ GGUF guardado en: {OUTPUT_DIR}/")

## 7. Descargar el GGUF a tu computadora

In [ ]:
import glob
from google.colab import files

# Descargar el archivo GGUF
gguf_files = glob.glob(f"{OUTPUT_DIR}/*.gguf")

for gguf_path in gguf_files:
    print(f"Descargando: {gguf_path}")
    files.download(gguf_path)

print("\n📦 Una vez descargado, en tu Mac:")
print(f'  echo "FROM ./{OUTPUT_DIR}-{QUANTIZATION}.gguf" > Modelfile')
print(f'  ollama create kai-{MODELO} -f Modelfile')
print(f'  ollama run kai-{MODELO}')

## 8. Opcional — Subir a HuggingFace Hub

In [ ]:
# Solo si querés publicar el modelo en HuggingFace
# HF_TOKEN = "hf_..."  # ← tu token de HuggingFace
# HF_USERNAME = "tu-usuario"

# model.push_to_hub_gguf(
#     f"{HF_USERNAME}/kai-{MODELO}-finetuned",
#     tokenizer,
#     quantization_method=QUANTIZATION,
#     token=HF_TOKEN,
# )
# print(f"✅ Modelo publicado en HuggingFace")

print("(celda comentada — descomentar para publicar en HuggingFace)")

---

## Recursos

- [Unsloth Studio](https://unsloth.ai/docs/new/studio) — para correr el GGUF en tu Mac sin código
- [Unsloth Qwen3.5 Guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune)
- [Unsloth Gemma 4 Guide](https://unsloth.ai/docs/models/gemma-4/train)
- [Kai — Comunidad AI LATAM](https://kaizen.la)

---

*Notebook creado por [Kai](https://kaizen.la). Código basado en la documentación oficial de Unsloth (MIT License).*